# Roman CS — T200 vs T500 ResMLP Analysis

**Plots in this notebook:**
- **Scaling curves** — f(Δχ²>0.2) vs N_train: single-prescription panels, side-by-side doubles, and all-4 combined
- **Δχ² distributions** — histograms at a chosen N_train: T200 vs T500, Taka vs Mead
- **Loss curves** — train/valid loss vs epoch: all N_trains per group, T200 vs T500, Taka vs Mead

**Cluster:** NvWulf · **Location:** `$ROOTDIR/projects/roman_real/`


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import h5py
import sys
from pathlib import Path

ROOTDIR = Path('/lustre/nvwulf/projects/MirandaGroup-nvwulf/bela/cocoa/Cocoa')
ROMAN   = ROOTDIR / 'projects' / 'roman_real'
sys.path.insert(0, str(ROOTDIR))
from emulator import ResMLP


In [ ]:
_TAKA = ROMAN / 'dvs' / 'taka'
_MEAD = ROMAN / 'dvs' / 'mead'

_TAKA_T100_DV  = _TAKA / 'roman_real_lcdm_b_taka_test_datavectors_T100.npy'
_TAKA_T100_PAR = _TAKA / 'roman_real_lcdm_b_taka_test_parameters_T100.txt'
_TAKA_T250_DV  = _TAKA / 'roman_real_lcdm_b_taka_test_datavectors_T250.npy'
_TAKA_T250_PAR = _TAKA / 'roman_real_lcdm_b_taka_test_parameters_T250.txt'
_MEAD_T100_DV  = _MEAD / 'roman_real_lcdm_b_mead_test_datavectors_T100.npy'
_MEAD_T100_PAR = _MEAD / 'roman_real_lcdm_b_mead_test_parameters_T100.txt'
_MEAD_T250_DV  = _MEAD / 'roman_real_lcdm_b_mead_test_datavectors_T250.npy'
_MEAD_T250_PAR = _MEAD / 'roman_real_lcdm_b_mead_test_parameters_T250.txt'


In [ ]:
SCRATCH_SIZES = [1_000, 2_500, 5_000, 7_500, 10_000, 25_000,
                 50_000, 100_000, 250_000, 500_000, 1_000_000]
TL_SIZES      = [1_000, 2_500, 5_000, 7_500, 10_000, 25_000,
                 50_000, 100_000, 250_000, 500_000, 1_000_000]

INCLUDE_TAKA_T200 = True
INCLUDE_TAKA_T500 = True
INCLUDE_MEAD_T200 = True
INCLUDE_MEAD_T500 = True
INCLUDE_TL_T200   = False   # flip True once TL runs complete
INCLUDE_TL_T500   = False


In [ ]:
_CHAINS = ROMAN / 'chains'

_CONFIGS = {
    'taka_T200': dict(
        model_dir    = _CHAINS / 'taka_scratch'      / 'models',
        model_prefix = 'roman_cs_lcdm_taka_resmlp_T200',
        test_dv=_TAKA_T100_DV, test_par=_TAKA_T100_PAR,
        n_dv=1080, sizes=SCRATCH_SIZES, shared_h5=None,
    ),
    'taka_T500': dict(
        model_dir    = _CHAINS / 'taka_scratch_T500' / 'models',
        model_prefix = 'roman_cs_lcdm_taka_resmlp_T500',
        test_dv=_TAKA_T250_DV, test_par=_TAKA_T250_PAR,
        n_dv=1080, sizes=SCRATCH_SIZES, shared_h5=None,
    ),
    'mead_T200': dict(
        model_dir    = _CHAINS / 'mead_scratch'      / 'models',
        model_prefix = 'roman_cs_lcdm_mead_resmlp_T200',
        test_dv=_MEAD_T100_DV, test_par=_MEAD_T100_PAR,
        n_dv=1080, sizes=SCRATCH_SIZES, shared_h5=None,
    ),
    'mead_T500': dict(
        model_dir    = _CHAINS / 'mead_scratch_T500' / 'models',
        model_prefix = 'roman_cs_lcdm_mead_resmlp_T500',
        test_dv=_MEAD_T250_DV, test_par=_MEAD_T250_PAR,
        n_dv=1080, sizes=SCRATCH_SIZES, shared_h5=None,
    ),
    # TL placeholders — update model_dir/prefix when runs land
    'tl_T200': dict(
        model_dir    = _CHAINS / 'tl_taka2mead_T200' / 'models',   # TODO
        model_prefix = 'roman_cs_lcdm_taka2mead_resmlp_TL_T200',   # TODO
        test_dv=_MEAD_T100_DV, test_par=_MEAD_T100_PAR,
        n_dv=1080, sizes=TL_SIZES, shared_h5=None,
    ),
    'tl_T500': dict(
        model_dir    = _CHAINS / 'tl_taka2mead_T500' / 'models',   # TODO
        model_prefix = 'roman_cs_lcdm_taka2mead_resmlp_TL_T500',   # TODO
        test_dv=_MEAD_T250_DV, test_par=_MEAD_T250_PAR,
        n_dv=1080, sizes=TL_SIZES, shared_h5=None,
    ),
}

OUTPUT_DIR = ROMAN / 'chains' / 'analysis'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Config loaded.')
for g, c in _CONFIGS.items():
    print(f"  {g}: model_dir exists={c['model_dir'].exists()}")


In [ ]:
# ============================================================================
# PLOT SETTINGS
# Color: orange=Taka  blue=Mead  purple=TL
# Line:  solid=T200   dashed=T500
# Marker: diamond (ResMLP throughout)
# ============================================================================
plt.rcParams['mathtext.fontset']    = 'stix'
plt.rcParams['font.family']         = 'STIXGeneral'
plt.rcParams['xtick.bottom']        = True
plt.rcParams['xtick.top']           = False
plt.rcParams['ytick.right']         = False
plt.rcParams['axes.edgecolor']      = 'black'
plt.rcParams['axes.linewidth']      = '1.0'
plt.rcParams['axes.grid']           = True
plt.rcParams['grid.alpha']          = 0.18
plt.rcParams['grid.color']          = 'lightgray'
plt.rcParams['legend.labelspacing'] = 0.5
plt.rcParams['savefig.bbox']        = 'tight'
plt.rcParams['savefig.format']      = 'pdf'
plt.rcParams['text.usetex']         = False
plt.rcParams.update({
    'font.size':        14,
    'axes.labelsize':   20,
    'axes.titlesize':   22,
    'xtick.labelsize':  16,
    'ytick.labelsize':  16,
    'legend.fontsize':  13,
    'figure.titlesize': 24,
})

GROUP_STYLES = {
    'taka_T200': dict(color='#E07B00', marker='D', linestyle='-'),
    'taka_T500': dict(color='#E07B00', marker='D', linestyle='--'),
    'mead_T200': dict(color='#1A6F9A', marker='D', linestyle='-'),
    'mead_T500': dict(color='#1A6F9A', marker='D', linestyle='--'),
    'tl_T200':   dict(color='#7B2D8B', marker='^', linestyle='-'),
    'tl_T500':   dict(color='#7B2D8B', marker='^', linestyle='--'),
}

GROUP_LABELS = {
    'taka_T200': r'Taka  $T{=}200$',
    'taka_T500': r'Taka  $T{=}500$',
    'mead_T200': r'Mead $T{=}200$',
    'mead_T500': r'Mead $T{=}500$',
    'tl_T200':   r'TL Taka$\to$Mead  $T{=}200$',
    'tl_T500':   r'TL Taka$\to$Mead  $T{=}500$',
}

print('Plot settings loaded.')


In [ ]:
# ============================================================================
# LOADING — Δχ² inference + disk cache
# Cache: chains/analysis/results_cache_T200vsT500.npz
# Recompute one entry: del results[group][n], re-run.
# Wipe all: delete the .npz file.
# ============================================================================
RESULTS_CACHE  = OUTPUT_DIR / 'results_cache_T200vsT500.npz'
_test_data_cache = {}

def get_test_data(test_dv, test_par, n_dv):
    key = (str(test_dv), str(test_par))
    if key not in _test_data_cache:
        print(f'    [cache miss] loading: {Path(test_par).name}')
        header = np.array(open(test_par).readline().split()[1:])
        x_raw  = torch.tensor(np.loadtxt(test_par), dtype=torch.float32)
        y_raw  = torch.tensor(np.load(test_dv)[:, :n_dv], dtype=torch.float32)
        _test_data_cache[key] = (x_raw, y_raw, header)
    return _test_data_cache[key]

def run_inference(group_key, n_train):
    cfg     = _CONFIGS[group_key]
    n_dv    = cfg['n_dv']
    pt_path = cfg['model_dir'] / f"{cfg['model_prefix']}_N{n_train}.pt"
    h5_path = cfg['shared_h5'] or (cfg['model_dir'] / f"{cfg['model_prefix']}_N{n_train}.h5")
    if not pt_path.exists():
        print(f'    [skip] {pt_path.name}')
        return None
    if not h5_path.exists():
        print(f'    [skip] {h5_path.name}')
        return None
    with h5py.File(h5_path) as f:
        s_mean  = torch.tensor(f['sample_mean'][:], dtype=torch.float32)
        s_std   = torch.tensor(f['sample_std'][:],  dtype=torch.float32)
        evals   = torch.tensor(f['dv_evals'][:],    dtype=torch.float32)
        evecs   = torch.tensor(f['dv_evecs'][:],    dtype=torch.float32)
        fid     = torch.tensor(f['dv_fid'][:],      dtype=torch.float32)
        tparams = [p.decode() if isinstance(p, bytes) else p for p in f['train_params'][:]]
    x_raw, y_raw, header = get_test_data(cfg['test_dv'], cfg['test_par'], n_dv)
    idxs        = [np.where(header == p)[0][0] for p in tparams]
    x_test      = (x_raw[:, idxs] - s_mean) / s_std
    y_test_norm = (y_raw[:, :n_dv] - fid) @ evecs / evals
    model = ResMLP(len(tparams), n_dv, 512)
    model.load_state_dict(torch.load(pt_path, map_location='cpu'))
    model.eval()
    with torch.no_grad():
        diff = y_test_norm - model(x_test)
    return (diff**2).sum(dim=1).numpy()

# load cache
if RESULTS_CACHE.exists():
    results = np.load(RESULTS_CACHE, allow_pickle=True)['results'].item()
    print(f'Loaded cache: {sum(len(v) for v in results.values())} entries')
else:
    results = {}
    print('No cache — will run inference.')

groups_to_load = [g for g, flag in [
    ('taka_T200', INCLUDE_TAKA_T200), ('taka_T500', INCLUDE_TAKA_T500),
    ('mead_T200', INCLUDE_MEAD_T200), ('mead_T500', INCLUDE_MEAD_T500),
    ('tl_T200',   INCLUDE_TL_T200),   ('tl_T500',   INCLUDE_TL_T500),
] if flag]

new_entries = 0
for group in groups_to_load:
    results.setdefault(group, {})
    print(f'\n=== {group.upper()} ===')
    for n in _CONFIGS[group]['sizes']:
        if n in results[group]:
            dc2 = results[group][n]
            print(f'  N={n:>9,}: [cached] median={np.median(dc2):.4f}  f>0.2={np.mean(dc2>0.2):.3f}')
            continue
        dc2 = run_inference(group, n)
        if dc2 is not None:
            results[group][n] = dc2
            new_entries += 1
            print(f'  N={n:>9,}: median={np.median(dc2):.4f}  f>0.2={np.mean(dc2>0.2):.3f}')

if new_entries > 0:
    np.savez(RESULTS_CACHE, results=results)
    print(f'\nSaved {new_entries} new entries.')
else:
    print('\nAll cached.')


---
## Scaling Curves — f(Δχ²>0.2) vs N_train

In [ ]:
def _scaling_ax(ax, groups, legend=True, legend_outside=False):
    """Draw scaling curves onto ax. Returns line handles for external legends."""
    handles = []
    for group in groups:
        if group not in results or not results[group]:
            continue
        sizes   = sorted(results[group].keys())
        sizes_k = [n / 1000 for n in sizes]
        frac    = [np.mean(results[group][n] > 0.2) for n in sizes]
        sty = GROUP_STYLES[group]
        ln, = ax.plot(sizes_k, frac, linewidth=2.5, markersize=8,
                      label=GROUP_LABELS[group],
                      color=sty['color'], marker=sty['marker'], linestyle=sty['linestyle'])
        handles.append(ln)
    ax.axhline(0.01, color='gray', linestyle=':',  linewidth=1.5, alpha=0.8, label='1%')
    ax.axhline(0.10, color='gray', linestyle='--', linewidth=1.5, alpha=0.8, label='10%')
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel(r'$N_{\rm train}\,/\,1000$')
    ax.set_ylabel(r'$f(\Delta\chi^2 > 0.2)$')
    ax.tick_params(direction='out', length=5, width=1.2, which='both')
    if legend and not legend_outside:
        ax.legend(fontsize=12, loc='upper right')
    return handles


In [ ]:
# ============================================================================
# SINGLE PANEL — Taka: T200 vs T500
# ============================================================================
fig, ax = plt.subplots(figsize=(7, 5.5))
_scaling_ax(ax, ['taka_T200', 'taka_T500'])
ax.set_title('Taka — $T{=}200$ vs $T{=}500$')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'scaling_taka_single.pdf', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# ============================================================================
# SINGLE PANEL — Mead: T200 vs T500
# ============================================================================
fig, ax = plt.subplots(figsize=(7, 5.5))
_scaling_ax(ax, ['mead_T200', 'mead_T500'])
ax.set_title('Mead — $T{=}200$ vs $T{=}500$')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'scaling_mead_single.pdf', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# ============================================================================
# DOUBLE PANEL — Taka (left) + Mead (right), T200 vs T500 on each
# ============================================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
_scaling_ax(axes[0], ['taka_T200', 'taka_T500'])
_scaling_ax(axes[1], ['mead_T200', 'mead_T500'])
axes[1].set_ylabel('')
axes[0].set_title('Taka')
axes[1].set_title('Mead')
fig.suptitle('ResMLP Scratch Baselines — $T{=}200$ vs $T{=}500$', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'scaling_double_byPrescription.pdf', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# ============================================================================
# DOUBLE PANEL — T200 (left) + T500 (right), Taka vs Mead on each
# ============================================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
_scaling_ax(axes[0], ['taka_T200', 'mead_T200'])
_scaling_ax(axes[1], ['taka_T500', 'mead_T500'])
axes[1].set_ylabel('')
axes[0].set_title('$T{=}200$')
axes[1].set_title('$T{=}500$')
fig.suptitle('Taka vs Mead ResMLP Scratch — by Temperature', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'scaling_double_byT.pdf', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# ============================================================================
# ALL 4 ON ONE PANEL — legend placed outside to avoid overlap
# ============================================================================
fig, ax = plt.subplots(figsize=(8, 5.5))
handles = _scaling_ax(ax, ['taka_T200', 'taka_T500', 'mead_T200', 'mead_T500'],
                      legend=False)
# add reference line handles manually for full legend
import matplotlib.lines as mlines
h1 = mlines.Line2D([], [], color='gray', linestyle=':',  linewidth=1.5, label='1%')
h2 = mlines.Line2D([], [], color='gray', linestyle='--', linewidth=1.5, label='10%')
ax.legend(handles=handles + [h1, h2],
          fontsize=11, loc='upper right',
          framealpha=0.9, edgecolor='gray',
          handlelength=2.5, labelspacing=0.4)
ax.set_title('ResMLP Scratch Baselines — All Cases')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'scaling_all4.pdf', dpi=200, bbox_inches='tight')
plt.show()


---
## Δχ² Distributions

In [ ]:
# ============================================================================
# CHOOSE N_TRAIN for chi2 distribution plots
# Must be a key that exists in results[group].
# ============================================================================
CHI2_N = 100_000   # <-- change this to compare different training sizes

def _chi2_ax(ax, groups, n_train, legend=True):
    """Draw Δχ² histograms for `groups` at fixed `n_train` onto ax."""
    bins = np.logspace(-3, 1.5, 60)
    for group in groups:
        if group not in results or n_train not in results[group]:
            print(f'[skip] {group} N={n_train:,} — not loaded')
            continue
        dc2 = results[group][n_train]
        sty = GROUP_STYLES[group]
        ax.hist(dc2, bins=bins, histtype='step', linewidth=2.0,
                label=GROUP_LABELS[group] + f'  (f={np.mean(dc2>0.2):.3f})',
                color=sty['color'],
                linestyle=sty['linestyle'] if sty['linestyle'] != '-' else 'solid')
    ax.axvline(0.2, color='red', linestyle='--', linewidth=1.5, alpha=0.8, label=r'$\Delta\chi^2=0.2$')
    ax.set_xscale('log')
    ax.set_xlabel(r'$\Delta\chi^2$')
    ax.set_ylabel('Count')
    ax.tick_params(direction='out', length=5, width=1.2, which='both')
    if legend:
        ax.legend(fontsize=11, loc='upper right')


In [ ]:
# ============================================================================
# CHI2 SINGLE — Taka T200 vs T500 at CHI2_N
# ============================================================================
fig, ax = plt.subplots(figsize=(7, 5.5))
_chi2_ax(ax, ['taka_T200', 'taka_T500'], CHI2_N)
ax.set_title(f'Taka — $T{{=}}200$ vs $T{{=}}500$ ($N_{{\\rm train}}={CHI2_N:,}$)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f'chi2_taka_N{CHI2_N}.pdf', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# ============================================================================
# CHI2 SINGLE — Mead T200 vs T500 at CHI2_N
# ============================================================================
fig, ax = plt.subplots(figsize=(7, 5.5))
_chi2_ax(ax, ['mead_T200', 'mead_T500'], CHI2_N)
ax.set_title(f'Mead — $T{{=}}200$ vs $T{{=}}500$ ($N_{{\\rm train}}={CHI2_N:,}$)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f'chi2_mead_N{CHI2_N}.pdf', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# ============================================================================
# CHI2 DOUBLE — Taka (left) + Mead (right), T200 vs T500
# ============================================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
_chi2_ax(axes[0], ['taka_T200', 'taka_T500'], CHI2_N)
_chi2_ax(axes[1], ['mead_T200', 'mead_T500'], CHI2_N)
axes[1].set_ylabel('')
axes[0].set_title('Taka')
axes[1].set_title('Mead')
fig.suptitle(f'$\\Delta\\chi^2$ distributions — $N_{{\\rm train}}={CHI2_N:,}$', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f'chi2_double_byPrescription_N{CHI2_N}.pdf', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# ============================================================================
# CHI2 DOUBLE — T200 (left) + T500 (right), Taka vs Mead
# ============================================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
_chi2_ax(axes[0], ['taka_T200', 'mead_T200'], CHI2_N)
_chi2_ax(axes[1], ['taka_T500', 'mead_T500'], CHI2_N)
axes[1].set_ylabel('')
axes[0].set_title('$T{=}200$')
axes[1].set_title('$T{=}500$')
fig.suptitle(f'$\\Delta\\chi^2$ distributions — $N_{{\\rm train}}={CHI2_N:,}$', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / f'chi2_double_byT_N{CHI2_N}.pdf', dpi=200, bbox_inches='tight')
plt.show()


---
## Loss Curves

In [ ]:
# ============================================================================
# LOSS LOADER
# losses.txt is saved by train_emulator.py as np.savetxt of shape (2, n_epochs):
#   row 0 = train loss per epoch
#   row 1 = valid loss per epoch
# ============================================================================
_losses_cache = {}

def load_losses(group_key, n_train):
    """Return (train_losses, valid_losses) arrays or (None, None) if file missing."""
    key = (group_key, n_train)
    if key in _losses_cache:
        return _losses_cache[key]
    cfg  = _CONFIGS[group_key]
    path = cfg['model_dir'] / f"{cfg['model_prefix']}_N{n_train}_losses.txt"
    if not path.exists():
        return None, None
    data = np.loadtxt(path)   # shape (2, n_epochs)
    _losses_cache[key] = (data[0], data[1])
    return data[0], data[1]


In [ ]:
# Color ramp for N_train — darker = more data
import matplotlib.cm as cm

def _n_color(n, all_sizes, base_color):
    """Lighten base_color for small N, full color for large N."""
    idx   = sorted(all_sizes).index(n)
    alpha = 0.25 + 0.75 * (idx / max(len(all_sizes) - 1, 1))
    c = plt.matplotlib.colors.to_rgba(base_color)
    # blend toward white for low alpha
    blended = tuple(alpha * c[i] + (1 - alpha) * 1.0 for i in range(3))
    return blended

def _loss_ax(ax, group_key, which='valid', sizes=None):
    """
    Plot loss curves for all available N_train in group onto ax.
    which: 'train', 'valid', or 'both'
    sizes: list of N_train to include; None = all in _CONFIGS
    """
    cfg        = _CONFIGS[group_key]
    base_color = GROUP_STYLES[group_key]['color']
    all_sizes  = sizes or cfg['sizes']
    plotted    = 0
    for n in all_sizes:
        tr, va = load_losses(group_key, n)
        if tr is None:
            continue
        c    = _n_color(n, all_sizes, base_color)
        lbl  = f'$N={n//1000}$k' if n < 1_000_000 else '$N=1$M'
        epochs = np.arange(1, len(tr) + 1)
        if which in ('train', 'both'):
            ax.plot(epochs, tr, color=c, linewidth=1.5,
                    linestyle='-', label=lbl if which == 'train' else f'{lbl} train')
        if which in ('valid', 'both'):
            ax.plot(epochs, va, color=c, linewidth=1.5,
                    linestyle='--' if which == 'both' else '-',
                    label=lbl if which == 'valid' else f'{lbl} valid')
        plotted += 1
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.set_yscale('log')
    ax.tick_params(direction='out', length=5, width=1.2)
    return plotted


In [ ]:
# ============================================================================
# LOSS SINGLE — Taka T200: all N_trains, validation loss
# ============================================================================
fig, ax = plt.subplots(figsize=(7, 5.5))
n = _loss_ax(ax, 'taka_T200', which='valid')
ax.legend(fontsize=10, ncol=2, loc='upper right')
ax.set_title('Taka $T{=}200$ — validation loss')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'loss_taka_T200.pdf', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# ============================================================================
# LOSS SINGLE — Taka T500: all N_trains, validation loss
# ============================================================================
fig, ax = plt.subplots(figsize=(7, 5.5))
_loss_ax(ax, 'taka_T500', which='valid')
ax.legend(fontsize=10, ncol=2, loc='upper right')
ax.set_title('Taka $T{=}500$ — validation loss')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'loss_taka_T500.pdf', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# ============================================================================
# LOSS SINGLE — Mead T200
# ============================================================================
fig, ax = plt.subplots(figsize=(7, 5.5))
_loss_ax(ax, 'mead_T200', which='valid')
ax.legend(fontsize=10, ncol=2, loc='upper right')
ax.set_title('Mead $T{=}200$ — validation loss')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'loss_mead_T200.pdf', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# ============================================================================
# LOSS SINGLE — Mead T500
# ============================================================================
fig, ax = plt.subplots(figsize=(7, 5.5))
_loss_ax(ax, 'mead_T500', which='valid')
ax.legend(fontsize=10, ncol=2, loc='upper right')
ax.set_title('Mead $T{=}500$ — validation loss')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'loss_mead_T500.pdf', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# ============================================================================
# LOSS DOUBLE — T200 vs T500: Taka (left) + Mead (right)
# Each panel overlays all N_trains for both T values.
# Solid = T200, dashed = T500 (same color family per N_train).
# ============================================================================
# Choose a representative subset of N_trains to keep it readable
LOSS_SIZES_SHOWN = [10_000, 50_000, 100_000, 250_000, 500_000, 1_000_000]

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

for ax, g200, g500, title in [
    (axes[0], 'taka_T200', 'taka_T500', 'Taka'),
    (axes[1], 'mead_T200', 'mead_T500', 'Mead'),
]:
    base_color = GROUP_STYLES[g200]['color']
    for n in LOSS_SIZES_SHOWN:
        c   = _n_color(n, LOSS_SIZES_SHOWN, base_color)
        lbl = f'$N={n//1000}$k' if n < 1_000_000 else '$N=1$M'
        tr200, va200 = load_losses(g200, n)
        tr500, va500 = load_losses(g500, n)
        if va200 is not None:
            epochs = np.arange(1, len(va200) + 1)
            ax.plot(epochs, va200, color=c, linewidth=1.8, linestyle='-',
                    label=f'{lbl}  $T{{=}}200$')
        if va500 is not None:
            epochs = np.arange(1, len(va500) + 1)
            ax.plot(epochs, va500, color=c, linewidth=1.8, linestyle='--',
                    label=f'{lbl}  $T{{=}}500$')
    ax.set_xlabel('Epoch')
    ax.set_yscale('log')
    ax.set_title(title)
    ax.tick_params(direction='out', length=5, width=1.2)
    ax.legend(fontsize=9, ncol=2, loc='upper right',
              framealpha=0.9, labelspacing=0.3)

axes[0].set_ylabel('Validation loss')
axes[1].set_ylabel('')
fig.suptitle('Validation loss — $T{=}200$ (solid) vs $T{=}500$ (dashed)', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'loss_double_T200vsT500.pdf', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# ============================================================================
# LOSS DOUBLE — Taka vs Mead: T200 (left) + T500 (right)
# Taka = orange, Mead = blue, darker = more data.
# ============================================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

for ax, g_taka, g_mead, title in [
    (axes[0], 'taka_T200', 'mead_T200', '$T{=}200$'),
    (axes[1], 'taka_T500', 'mead_T500', '$T{=}500$'),
]:
    for group in [g_taka, g_mead]:
        base_color = GROUP_STYLES[group]['color']
        ls         = GROUP_STYLES[group]['linestyle']
        for n in LOSS_SIZES_SHOWN:
            c   = _n_color(n, LOSS_SIZES_SHOWN, base_color)
            lbl = f'$N={n//1000}$k' if n < 1_000_000 else '$N=1$M'
            pres = 'Taka' if 'taka' in group else 'Mead'
            _, va = load_losses(group, n)
            if va is not None:
                epochs = np.arange(1, len(va) + 1)
                ax.plot(epochs, va, color=c, linewidth=1.8, linestyle=ls,
                        label=f'{pres} {lbl}')
    ax.set_xlabel('Epoch')
    ax.set_yscale('log')
    ax.set_title(title)
    ax.tick_params(direction='out', length=5, width=1.2)
    ax.legend(fontsize=9, ncol=2, loc='upper right',
              framealpha=0.9, labelspacing=0.3)

axes[0].set_ylabel('Validation loss')
axes[1].set_ylabel('')
fig.suptitle('Validation loss — Taka (solid) vs Mead (dashed)', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'loss_double_TakaVsMead.pdf', dpi=200, bbox_inches='tight')
plt.show()


---
## Summary Table

In [ ]:
print(f"{'Group':<16} {'N_train':>10}  {'Median Δχ²':>12}  {'f(>0.2)':>9}")
print('-' * 54)
for group in ['taka_T200', 'taka_T500', 'mead_T200', 'mead_T500', 'tl_T200', 'tl_T500']:
    if group not in results:
        continue
    for n in sorted(results[group].keys()):
        dc2 = results[group][n]
        print(f"{group:<16} {n:>10,}  {np.median(dc2):>12.4f}  {np.mean(dc2>0.2):>9.4f}")
    print()
